In [1]:
%pip install -q langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter LangSmith API Key: ")
    
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'rag-lab-query-translation'

In [3]:
# Loading the Vectorstore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma   
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini")
)

try:
    result = embeddings.embed_query("What are ChebConv layers?")
    print("SUCCESS, vector length:", len(result))
except Exception as e:
    print("FAILED:", e)
    
print("Chunks in store:", vectorstore._collection.count())

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
#llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

SUCCESS, vector length: 3072
Chunks in store: 1555


In [4]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

step_back_examples = [
    {
        "input": "What k-factor did Decke et al. use for their ChebConv layers?",
        "output": "What are ChebConv layers and how does the k-factor parameter affect them?",
    },
    {
        "input": "Does Qian et al.'s pressure normalization scheme apply during the ghost-layer exchange or during final output?",
        "output": "How is pressure indeterminacy handled in distributed PINN training?",
    },
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}"),
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=step_back_examples,
)

STEP_BACK_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an expert at reframing questions. Your task is to step "
     "back and paraphrase a specific question into a more generic, "
     "conceptual question that is easier to find background context "
     "for. Here are a few examples:"),
    few_shot_prompt,
    ("human", "{question}"),
])

In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStore
from typing import List


class StepBackStrategy:
    """Generates a more abstract/general version of the query, retrieves
    using BOTH the original specific query and the step-back query, and
    answers using the combined context from both."""

    def __init__(self, vectorstore: VectorStore, llm: BaseChatModel, k: int = 4):
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        self.llm = llm
        self.step_back_chain = STEP_BACK_PROMPT | llm | StrOutputParser()

        self.answer_prompt = ChatPromptTemplate.from_template(
            "You are an expert at answering technical questions.\n\n"
            "Here is context from the specific question:\n{normal_context}\n\n"
            "Here is broader background context from a more general "
            "version of the question:\n{step_back_context}\n\n"
            "Original question: {question}\n\n"
            "Using the context above, answer the original question."
        )

    def generate_step_back_query(self, query: str) -> str:
        return self.step_back_chain.invoke({"question": query})

    def retrieve(self, query: str) -> List[Document]:
        """Returns the union of both retrievals — specific-query docs
        first, step-back-query docs after, for eval harness inspection."""
        step_back_query = self.generate_step_back_query(query)
        normal_docs = self.retriever.invoke(query)
        step_back_docs = self.retriever.invoke(step_back_query)
        return normal_docs + step_back_docs

    def run(self, query: str) -> str:
        step_back_query = self.generate_step_back_query(query)
        normal_docs = self.retriever.invoke(query)
        step_back_docs = self.retriever.invoke(step_back_query)

        normal_context = "\n\n".join(d.page_content for d in normal_docs)
        step_back_context = "\n\n".join(d.page_content for d in step_back_docs)

        chain = self.answer_prompt | self.llm | StrOutputParser()
        return chain.invoke({
            "normal_context": normal_context,
            "step_back_context": step_back_context,
            "question": query,
        })

In [6]:
from rag_lab.strategies.step_back import StepBackStrategy

strategy = StepBackStrategy(vectorstore, llm=llm)
test_query = "What k-factor did Decke et al. use for their ChebConv layers?"

In [7]:
try:
    step_back_q = strategy.generate_step_back_query(test_query)
    print("Step-back query generation OK:", step_back_q)
except Exception as e:
    print("FAILED at step-back generation:", e)

Step-back query generation OK: What considerations guide the selection of the polynomial order (k-factor) for Chebyshev graph convolution (ChebConv) layers in graph neural network architectures?


In [8]:
try:
    normal_docs = strategy.retriever.invoke(test_query)
    print("Normal retrieval OK")
except Exception as e:
    print("FAILED at normal retrieval:", e)
import time

for attempt in range(3):
    try:
        step_back_docs = strategy.retriever.invoke(step_back_q)
        print(f"Succeeded on attempt {attempt + 1}")
        break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        time.sleep(3)

Normal retrieval OK
Succeeded on attempt 1


In [9]:
from rag_lab.strategies.step_back import StepBackStrategy

test_query = "What k-factor did Decke et al. use for their ChebConv layers?"
strategy = StepBackStrategy(vectorstore, llm=llm)

step_back_q = strategy.generate_step_back_query(test_query)
print(f"Original: {test_query}")
print(f"Step-back: {step_back_q}\n")

print(strategy.run(test_query))

Original: What k-factor did Decke et al. use for their ChebConv layers?
Step-back: What considerations guide the choice of the Chebyshev polynomial order (the k-factor) when configuring ChebConv layers in graph neural networks?

Decke et al. set the **k‑factor to 6** for their ChebConv (k‑hop) layers. In other words, each ChebConv aggregated information from nodes up to six hops away, which they found helped the model capture larger structures and reduce prediction error.
